# Geothermal AI — 1307 Colab runner

Run Moraga-style training (`doe_geoai.py`) or tile generation (`create_doe_dataset.py`) in Google Colab.

- Training code is loaded from GitHub (`Git_1307/` in this repository).
- Heavy data (multi-band `.gri` stacks, tile `.tar.gz` archives) is pulled from Google Cloud Storage after you authenticate and sync the prefixes listed in configuration.
- Run artifacts (model, labels, metrics, `run_config.json`) are written under `/content/1307_runs/…` on the VM. Use `gsutil` or the Cloud Console to copy them off ephemeral disk before the session ends.

Run sections in order from §1 onward after opening this notebook from GitHub or uploading it to Colab.


### Optional: `gcloud` / `gsutil` on your computer

For syncing from your laptop (e.g. PowerShell) into the same bucket Colab uses.

1. Install [Google Cloud CLI](https://cloud.google.com/sdk/docs/install) (includes `gcloud` and `gsutil`).
2. Restart the terminal or IDE so `PATH` includes the SDK `bin` folder.
3. Verify: `gcloud --version` and `gsutil version`.
4. On a new machine: `gcloud auth login`, then `gcloud config set project YOUR_PROJECT_ID` ([Cloud Console](https://console.cloud.google.com/) project ID, not the display name).

Colab uses a separate in-session Google sign-in. Keep `GCP_PROJECT` in §1 aligned with that project.


## 1) Configure

Set bucket, Git clone settings, data prefixes, and job variables. Re-run this cell whenever you change them.


In [ ]:
GCP_PROJECT = "maloney-geog-473"
GCS_BUCKET = "gis-final-project"

# GitHub: repository that contains Git_1307/ (fork or mirror if needed).
GIT_REPO_URL = "https://github.com/GarretMaloney/GeothermalAI.git"
GIT_BRANCH = "main"
LOCAL_REPO_DIR = "/content/GeothermalAI"
CODE_SUBDIR = "Git_1307"
LOCAL_1307_DIR = f"{LOCAL_REPO_DIR.rstrip('/')}/{CODE_SUBDIR}"

# Large inputs: each prefix is rsynced from gs://BUCKET/... into /content/...
GCS_EXTRA_SYNC_PREFIXES = [
    "GIS Final Project/BradyGDB/BradyRaw/Brady_Analysis/Geophysics/BradySOM",
]

REQUIREMENTS_FILE = "requirements.txt"
EXTRA_PIP_PACKAGES = ""  # optional: extra pip packages, space-separated

# Entry point: "doe_geoai.py" or "create_doe_dataset.py"
ENTRY_SCRIPT = "doe_geoai.py"
ENTRY_ARGS = ""  # blank: build arguments from DOE_* below

DOE_GRI_INPUT = "/content/GIS Final Project/BradyGDB/BradyRaw/Brady_Analysis/Geophysics/BradySOM/brady_som_output.gri"
DOE_DATASET_OUT_DIR = "/content/doe-data/brady_samples_19x3d"
DOE_CHANNELS = 3
DOE_SAMPLE_COUNT = 100000
DOE_KERNEL_PIXELS = 19

DOE_DATASET_PATH = "/content/doe-data/brady_samples_19x3d"
AUTO_DOWNLOAD_DATASET_FROM_GCS = True
GCS_DATASET_ARCHIVE = "GIS Final Project/outputs/doe-datasets/brady_samples_19x3d.tar.gz"
DOE_LABELBIN_PATH = ""
DOE_MODEL_PATH = ""
DOE_PLOT_PATH = ""
DOE_CURVES_PATH = ""

# Training hyperparameters (Moraga et al. use 100 epochs with augmentation at paper fidelity).
DOE_EPOCHS = 25
DOE_BATCH_SIZE = 32
DOE_GPUS = 1
DOE_EXTRA_ARGS = ""

# After create_doe_dataset: upload one .tar.gz here when SYNC_DATASET_TO_GCS is True.
GCS_DATASET_PREFIX = "GIS Final Project/outputs/doe-datasets"
SYNC_DATASET_TO_GCS = True

# Run-folder upload (optional): §6 builds GCS_RUN_URI = gs://GCS_BUCKET/{GCS_OUTPUT_PREFIX}/{run_name}.
GCS_OUTPUT_PREFIX = "GIS Final Project/outputs/1307"
RUN_NAME_OVERRIDE = ""  # blank: timestamp under /content/1307_runs/

AUTO_APPEND_OUTPUT_ARGS = False
OUTPUT_DIR_FLAG = "--output_dir"
SAVE_DIR_FLAG = "--save_dir"


## 2) Authenticate; clone or update from GitHub; sync data from GCS

Authenticates with Google, updates the clone at `LOCAL_REPO_DIR` on branch `GIT_BRANCH`, then rsyncs each `GCS_EXTRA_SYNC_PREFIXES` path into `/content/...`.


In [ ]:
import shlex
import subprocess
from pathlib import Path

from google.colab import auth


def run(cmd, cwd=None):
    print("$", " ".join(shlex.quote(str(c)) for c in cmd))
    p = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr)
        raise subprocess.CalledProcessError(p.returncode, cmd, output=p.stdout, stderr=p.stderr)
    return p


if not GCP_PROJECT or not GCS_BUCKET:
    raise ValueError("Set GCP_PROJECT and GCS_BUCKET in the config cell first.")

auth.authenticate_user()
run(["gcloud", "config", "set", "project", GCP_PROJECT])

if not GIT_REPO_URL or not str(GIT_REPO_URL).strip():
    raise ValueError("Set GIT_REPO_URL in the config cell.")
repo_path = Path(LOCAL_REPO_DIR)
git_dir = repo_path / ".git"
if git_dir.is_dir():
    run(["git", "-C", str(repo_path), "fetch", "origin", GIT_BRANCH])
    run(["git", "-C", str(repo_path), "checkout", GIT_BRANCH])
    run(["git", "-C", str(repo_path), "pull", "--ff-only", "origin", GIT_BRANCH])
    print("Updated git repo at", repo_path)
else:
    if repo_path.exists() and any(repo_path.iterdir()):
        raise RuntimeError(
            f"{repo_path} exists but is not a git clone. Remove it in Colab (e.g. !rm -rf …) or change LOCAL_REPO_DIR."
        )
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO_URL.strip(), str(repo_path)])
    print("Cloned", GIT_REPO_URL, "->", repo_path)

code_dir = Path(LOCAL_1307_DIR)
if not code_dir.is_dir():
    raise FileNotFoundError(f"After git sync, code dir is missing: {code_dir} (check CODE_SUBDIR).")
print("Using training code from", code_dir)

for extra_prefix in GCS_EXTRA_SYNC_PREFIXES:
    ep = extra_prefix.strip("/")
    if not ep:
        continue
    extra_src = f"gs://{GCS_BUCKET}/{ep}"
    extra_dst = Path("/content") / ep
    extra_dst.mkdir(parents=True, exist_ok=True)
    try:
        run(["gsutil", "-m", "rsync", "-r", extra_src, str(extra_dst)])
        print("Synced extra prefix via rsync:", extra_src, "->", extra_dst)
    except subprocess.CalledProcessError as exc:
        print("rsync failed for", extra_src)
        if exc.stderr:
            print(exc.stderr)
        print("Falling back to gsutil cp -r for this prefix...")
        try:
            run(["gsutil", "-m", "cp", "-r", extra_src, str(extra_dst.parent)])
            print("Synced extra prefix via cp -r:", extra_src, "->", extra_dst.parent)
        except subprocess.CalledProcessError as exc2:
            if exc2.stderr:
                print(exc2.stderr)
            raise RuntimeError(
                f"Failed syncing {extra_src}. "
                "Narrow GCS_EXTRA_SYNC_PREFIXES and rerun this cell."
            ) from exc2


## 3) Inspect `Git_1307`

Lists top-level files under the training code directory (sanity check after clone).


In [ ]:
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing local code directory: {root}")

items = sorted(p.name for p in root.iterdir())
print(f"Top-level files/folders in {root}:")
for name in items[:200]:
    print(" -", name)

## 4) Install dependencies

Installs from `Git_1307/requirements.txt` when it exists; otherwise from the repo root `requirements.txt`.


In [ ]:
import sys
import shlex
from pathlib import Path

req_path = Path(LOCAL_1307_DIR) / REQUIREMENTS_FILE
req_repo = Path(LOCAL_REPO_DIR) / "requirements.txt"

run([sys.executable, "-m", "pip", "install", "-U", "pip"])
if req_path.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(req_path)])
elif req_repo.is_file():
    print("Using repo root requirements:", req_repo)
    run([sys.executable, "-m", "pip", "install", "-r", str(req_repo)])
else:
    print(f"No requirements at {req_path} or {req_repo}")

extra = EXTRA_PIP_PACKAGES.strip()
if extra:
    run([sys.executable, "-m", "pip", "install", *shlex.split(extra)])


## 5) Check GPU runtime

For faster training, use Colab: Runtime → Change runtime type → GPU (when available).


In [ ]:
import subprocess

try:
    run(["nvidia-smi"])
except (subprocess.CalledProcessError, FileNotFoundError, OSError):
    print("nvidia-smi not available. In Colab: Runtime -> Change runtime type -> GPU if you need a GPU.")

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
except Exception as e:
    print("Torch check skipped:", e)

## 6) Create a run folder

Defines `LOCAL_RUN_DIR` under `/content/1307_runs/` and the matching `GCS_RUN_URI` under `GCS_OUTPUT_PREFIX` (same pattern as the old §9 sync cell). §7 writes outputs into `LOCAL_RUN_DIR`.


In [ ]:
from datetime import datetime
from pathlib import Path

run_name = RUN_NAME_OVERRIDE.strip() or datetime.now().strftime("run_%Y%m%d_%H%M%S")
LOCAL_RUN_DIR = Path(f"/content/1307_runs/{run_name}")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

_output_prefix = GCS_OUTPUT_PREFIX.strip().strip("/")
if not _output_prefix:
    raise ValueError("Set GCS_OUTPUT_PREFIX in the config cell.")
GCS_RUN_URI = f"gs://{GCS_BUCKET}/{_output_prefix}/{run_name}"

print("Run name:", run_name)
print("Local run dir:", LOCAL_RUN_DIR)
print("GCS run uri:", GCS_RUN_URI)


### Before §7: prepare local inputs

Run after §6. Behavior depends on `ENTRY_SCRIPT`:

- `create_doe_dataset.py` — Ensures `DOE_GRI_INPUT` exists; if the path is under `/content/` and missing, rsyncs the parent prefix from `GCS_BUCKET`.
- `doe_geoai.py` — Ensures `DOE_DATASET_PATH` exists; can download `GCS_DATASET_ARCHIVE` and extract. For validation from a prior run, set `PREP_MODEL_GCS_URI` / `PREP_LABELBIN_GCS_URI` to copy `.h5` / `.l` into `DOE_MODEL_PATH` / `DOE_LABELBIN_PATH`.

Leave the URIs blank when training from scratch (artifacts are produced under `LOCAL_RUN_DIR` in §7).


In [ ]:
from pathlib import Path
import shutil

# Optional knobs for this prep cell. Usually leave these as-is.
PREP_FORCE_REFRESH_DATASET = False  # True removes DOE_DATASET_PATH before re-extracting from GCS_DATASET_ARCHIVE.
PREP_MODEL_GCS_URI = ""  # Example: "gs://.../train_brady_19x5d_100ep/doe_geoai_model.h5"
PREP_LABELBIN_GCS_URI = ""  # Example: "gs://.../train_brady_19x5d_100ep/doe_labels.l"


def _gcs_uri(prefix_or_uri):
    prefix_or_uri = str(prefix_or_uri).strip()
    if not prefix_or_uri:
        return ""
    if prefix_or_uri.startswith("gs://"):
        return prefix_or_uri
    return f"gs://{GCS_BUCKET}/{prefix_or_uri.strip('/')}"


def _copy_from_gcs_if_needed(gcs_uri, local_path, label):
    local_path = Path(str(local_path).strip())
    if local_path.exists():
        print(f"{label} exists:", local_path)
        return
    if not gcs_uri.strip():
        print(f"{label} missing and no GCS URI was provided:", local_path)
        return
    local_path.parent.mkdir(parents=True, exist_ok=True)
    run(["gsutil", "cp", _gcs_uri(gcs_uri), str(local_path)])
    print(f"Copied {label}:", local_path)


entry_name = Path(ENTRY_SCRIPT).name

if entry_name == "create_doe_dataset.py":
    gri_path = Path(DOE_GRI_INPUT.strip())
    if not gri_path.exists() and str(gri_path).startswith("/content/"):
        # The notebook mirrors GCS paths under /content, so derive the parent GCS prefix.
        rel_parent = gri_path.parent.relative_to("/content").as_posix()
        source_uri = f"gs://{GCS_BUCKET}/{rel_parent}"
        gri_path.parent.mkdir(parents=True, exist_ok=True)
        print("DOE_GRI_INPUT missing; syncing parent prefix:", source_uri)
        run(["gsutil", "-m", "rsync", "-r", source_uri, str(gri_path.parent)])
    print("DOE_GRI_INPUT exists:", gri_path.exists(), gri_path)

elif entry_name == "doe_geoai.py":
    dataset_path = Path(DOE_DATASET_PATH.strip())
    if PREP_FORCE_REFRESH_DATASET and dataset_path.exists():
        print("Removing existing dataset folder:", dataset_path)
        shutil.rmtree(dataset_path)

    if not dataset_path.exists() and AUTO_DOWNLOAD_DATASET_FROM_GCS:
        archive_prefix = GCS_DATASET_ARCHIVE.strip()
        if not archive_prefix:
            archive_prefix = f"{GCS_DATASET_PREFIX.strip().strip('/')}/{dataset_path.name}.tar.gz"
        archive_uri = _gcs_uri(archive_prefix)
        local_archive = dataset_path.parent / Path(archive_prefix).name
        dataset_path.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading dataset archive:", archive_uri)
        run(["gsutil", "cp", archive_uri, str(local_archive)])
        print("Extracting dataset archive to:", dataset_path.parent)
        run(["tar", "-xzf", str(local_archive), "-C", str(dataset_path.parent)])

    print("DOE_DATASET_PATH exists:", dataset_path.exists(), dataset_path)

    if DOE_MODEL_PATH.strip():
        _copy_from_gcs_if_needed(PREP_MODEL_GCS_URI, DOE_MODEL_PATH, "DOE_MODEL_PATH")
    else:
        print("DOE_MODEL_PATH blank; §7 will write model under LOCAL_RUN_DIR for training.")

    if DOE_LABELBIN_PATH.strip():
        _copy_from_gcs_if_needed(PREP_LABELBIN_GCS_URI, DOE_LABELBIN_PATH, "DOE_LABELBIN_PATH")
    else:
        print("DOE_LABELBIN_PATH blank; §7 will write labels under LOCAL_RUN_DIR for training.")

else:
    print("No specific prep checks for ENTRY_SCRIPT:", ENTRY_SCRIPT)

## 7) Run metadata and entry script

Writes `run_config.json` (environment, git revision, paths, command) and runs `ENTRY_SCRIPT`. When `ENTRY_ARGS` is empty, arguments are built from the `DOE_*` settings.

- `create_doe_dataset.py` — Uses `DOE_GRI_INPUT`, `DOE_DATASET_OUT_DIR`, `DOE_CHANNELS`, `DOE_SAMPLE_COUNT`, `DOE_KERNEL_PIXELS`.
- `doe_geoai.py` — Uses `DOE_DATASET_PATH` and writes model, label bin, plots, and curves under `LOCAL_RUN_DIR` unless paths are overridden.

Applies small patches to `doe_tiff`, `create_doe_dataset.py`, and `doe_geoai.py` for Colab and TensorFlow Keras compatibility.

If `SYNC_DATASET_TO_GCS` is True and the entry script is `create_doe_dataset.py`, the tile folder is packed into one `.tar.gz` under `GCS_DATASET_PREFIX` at the end of this cell.


In [ ]:
import json
import os
import platform
import re
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

args = ENTRY_ARGS.strip()

entry = Path(LOCAL_1307_DIR) / ENTRY_SCRIPT
if not entry.exists():
    raise FileNotFoundError(
        f"ENTRY_SCRIPT not found: {entry}\n"
        "Update ENTRY_SCRIPT in the config cell."
    )

# Compatibility patch: synced 1307 tree has `doe_tiff/doe_tiff/*.py` but nested
# `__init__.py` used absolute imports (`from doe_tiff.io ...`) that break when only
# the nested package exists. Fix that, then import the nested package as `dt`.
root1307 = Path(LOCAL_1307_DIR)
nested_init = root1307 / "doe_tiff" / "doe_tiff" / "__init__.py"
if nested_init.is_file():
    init_txt = nested_init.read_text(encoding="utf-8", errors="ignore")
    init_patched = (
        init_txt.replace("from doe_tiff.io import", "from .io import")
        .replace("from doe_tiff.doe_kernel import", "from .doe_kernel import")
    )
    if init_patched != init_txt:
        nested_init.write_text(init_patched, encoding="utf-8")
        print("Patched doe_tiff/doe_tiff/__init__.py with relative imports")

if entry.name == "create_doe_dataset.py":
    src = entry.read_text(encoding="utf-8", errors="ignore")
    orig = src

    bad_block = (
        "try:\n"
        "    from doe_tiff.io import read_gdal_file, frame_image\n"
        "    from doe_tiff.doe_kernel import GeoTiffConvolution\n"
        "except Exception:\n"
        "    from doe_tiff.doe_tiff.io import read_gdal_file, frame_image\n"
        "    from doe_tiff.doe_tiff.doe_kernel import GeoTiffConvolution\n"
    )
    if bad_block in src:
        src = src.replace(bad_block, "")

    src = src.replace("import doe_tiff as dt", "import doe_tiff.doe_tiff as dt")
    src = src.replace("dt.io.read_gdal_file(", "dt.read_gdal_file(")

    # If earlier patches stripped `dt.` from helpers, restore it safely.
    src = re.sub(r"(?<!\.)read_gdal_file\(", "dt.read_gdal_file(", src)
    src = re.sub(r"(?<!\.)frame_image\(", "dt.frame_image(", src)
    src = re.sub(r"(?<!\.)GeoTiffConvolution\(", "dt.GeoTiffConvolution(", src)

    if src != orig:
        entry.write_text(src, encoding="utf-8")
        print("Patched create_doe_dataset.py for nested doe_tiff package layout")

if entry.name == "doe_geoai.py":
    dg = entry.read_text(encoding="utf-8", errors="ignore")
    dg0 = dg
    dg = re.sub(r"(?m)^\s*import\s+keras\s*$", "from tensorflow import keras", dg)
    dg = re.sub(r"(?m)^\s*from\s+keras\.callbacks\s+import\s+", "from tensorflow.keras.callbacks import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.layers\.convolutional\s+import\s+",
        "from tensorflow.keras.layers import ",
        dg,
    )
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.layers\.core\s+import\s+",
        "from tensorflow.keras.layers import ",
        dg,
    )
    dg = re.sub(r"(?m)^\s*from\s+keras\.layers\s+import\s+", "from tensorflow.keras.layers import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.regularizers\s+import\s+",
        "from tensorflow.keras.regularizers import ",
        dg,
    )
    dg = re.sub(r"(?m)^\s*from\s+keras\.models\s+import\s+", "from tensorflow.keras.models import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.optimizers\s+import\s+",
        "from tensorflow.keras.optimizers import ",
        dg,
    )
    dg = re.sub(r"(?m)^\s*from\s+keras\.utils\s+import\s+", "from tensorflow.keras.utils import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\s+import\s+backend\s+as\s+K\s*$",
        "from tensorflow.keras import backend as K",
        dg,
    )
    dg = re.sub(r"\bnp\.float\b", "float", dg)
    dg = re.sub(r"\bnp\.int\b", "int", dg)
    dg = re.sub(r"\bnp\.bool\b", "bool", dg)
    if "from tensorflow.keras.utils import multi_gpu_model" in dg and "def multi_gpu_model(model, gpus=None):" not in dg:
        dg = dg.replace(
            "from tensorflow.keras.utils import multi_gpu_model",
            "try:\n    from tensorflow.keras.utils import multi_gpu_model\nexcept Exception:\n    def multi_gpu_model(model, gpus=None):\n        return model",
        )
    dg = re.sub(
        r"ROC_curve_calc\(\s*testY\s*,\s*pre_y2\s*,\s*class_num\s*=\s*8\s*,",
        "ROC_curve_calc( testY, pre_y2, class_num=int(pre_y2_prob.shape[1]),",
        dg,
    )
    if dg != dg0:
        entry.write_text(dg, encoding="utf-8")
        print("Patched doe_geoai.py for tensorflow.keras + ROC class count on Colab")

cmd = [sys.executable, str(entry)]

# Auto-build args when ENTRY_ARGS is left blank.
if not args and entry.name == "create_doe_dataset.py":
    gri = DOE_GRI_INPUT.strip()
    if not gri:
        raise ValueError("Set DOE_GRI_INPUT in the config cell to your .gri path.")
    auto_args = [
        "-i", gri,
        "-c", str(DOE_CHANNELS),
        "-d", DOE_DATASET_OUT_DIR,
        "-s", str(DOE_SAMPLE_COUNT),
        "-k", str(DOE_KERNEL_PIXELS),
    ]
    cmd.extend(auto_args)
elif not args and entry.name == "doe_geoai.py":
    if not DOE_DATASET_PATH.strip():
        raise ValueError(
            "doe_geoai.py needs a training dataset. Set DOE_DATASET_PATH to the extracted tile folder."
        )

    dataset_path = Path(DOE_DATASET_PATH.strip())
    if AUTO_DOWNLOAD_DATASET_FROM_GCS and not dataset_path.exists():
        archive_prefix = GCS_DATASET_ARCHIVE.strip()
        if not archive_prefix:
            archive_prefix = f"{GCS_DATASET_PREFIX.strip().strip('/')}/{dataset_path.name}.tar.gz"
        archive_uri = archive_prefix if archive_prefix.startswith("gs://") else f"gs://{GCS_BUCKET}/{archive_prefix.strip('/')}"
        local_archive = dataset_path.parent / Path(archive_prefix).name
        dataset_path.parent.mkdir(parents=True, exist_ok=True)
        print("Dataset folder missing; downloading:", archive_uri)
        run(["gsutil", "cp", archive_uri, str(local_archive)])
        print("Extracting", local_archive, "to", dataset_path.parent)
        run(["tar", "-xzf", str(local_archive), "-C", str(dataset_path.parent)])

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Training dataset folder not found after download/extract: {dataset_path}. "
            "Check DOE_DATASET_PATH and GCS_DATASET_ARCHIVE."
        )

    dataset = str(dataset_path)
    labelbin = DOE_LABELBIN_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_labels.l")
    model = DOE_MODEL_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_model.h5")
    plot = DOE_PLOT_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_plot.png")
    curves = DOE_CURVES_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_curves.csv")

    auto_args = [
        "-d", dataset,
        "-l", labelbin,
        "-m", model,
        "-p", plot,
        "-o", curves,
        "-e", str(DOE_EPOCHS),
        "-b", str(DOE_BATCH_SIZE),
        "-g", str(DOE_GPUS),
        "-k", str(DOE_KERNEL_PIXELS),
        "-c", str(DOE_CHANNELS),
    ]
    extra = DOE_EXTRA_ARGS.strip()
    if extra:
        auto_args.extend(shlex.split(extra))

    cmd.extend(auto_args)
else:
    if args:
        cmd.extend(shlex.split(args))

if AUTO_APPEND_OUTPUT_ARGS:
    cmd.extend([
        OUTPUT_DIR_FLAG,
        str(LOCAL_RUN_DIR),
        SAVE_DIR_FLAG,
        str(LOCAL_RUN_DIR / "checkpoints"),
    ])

# Write a reproducible run manifest before execution.
_git_commit = None
if Path(LOCAL_REPO_DIR, ".git").is_dir():
    try:
        _cp = subprocess.run(
            ["git", "-C", str(Path(LOCAL_REPO_DIR)), "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        _git_commit = _cp.stdout.strip()
    except Exception as _e:
        _git_commit = f"(error: {_e})"
else:
    _git_commit = None

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "git_repo_url": GIT_REPO_URL,
    "git_branch": GIT_BRANCH,
    "git_commit": _git_commit,
    "local_repo_dir": LOCAL_REPO_DIR,
    "entry_script": ENTRY_SCRIPT,
    "entry_args": args,
    "command": cmd,
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "gcp_project": GCP_PROJECT,
    "gcs_bucket": GCS_BUCKET,
    "gcs_output_prefix": GCS_OUTPUT_PREFIX,
    "gcs_run_uri": GCS_RUN_URI,
    "sync_dataset_to_gcs": SYNC_DATASET_TO_GCS,
    "gcs_dataset_prefix": GCS_DATASET_PREFIX,
    "local_code_dir": str(LOCAL_1307_DIR),
    "local_run_dir": str(LOCAL_RUN_DIR),
    "auto_append_output_args": AUTO_APPEND_OUTPUT_ARGS,
    "output_dir_flag": OUTPUT_DIR_FLAG,
    "save_dir_flag": SAVE_DIR_FLAG,
    "doe_gri_input": DOE_GRI_INPUT,
    "doe_dataset_out_dir": DOE_DATASET_OUT_DIR,
    "doe_channels": DOE_CHANNELS,
    "doe_sample_count": DOE_SAMPLE_COUNT,
    "doe_kernel_pixels": DOE_KERNEL_PIXELS,
    "doe_dataset_path": DOE_DATASET_PATH,
    "auto_download_dataset_from_gcs": AUTO_DOWNLOAD_DATASET_FROM_GCS,
    "gcs_dataset_archive": GCS_DATASET_ARCHIVE,
    "doe_labelbin_path": DOE_LABELBIN_PATH,
    "doe_model_path": DOE_MODEL_PATH,
    "doe_plot_path": DOE_PLOT_PATH,
    "doe_curves_path": DOE_CURVES_PATH,
    "doe_epochs": DOE_EPOCHS,
    "doe_batch_size": DOE_BATCH_SIZE,
    "doe_gpus": DOE_GPUS,
    "doe_extra_args": DOE_EXTRA_ARGS,
}
manifest_path = Path(LOCAL_RUN_DIR) / "run_config.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote", manifest_path)

run(cmd, cwd=LOCAL_1307_DIR)

if entry.name == "create_doe_dataset.py" and SYNC_DATASET_TO_GCS and (not args):
    out_dir = Path(DOE_DATASET_OUT_DIR).resolve()
    pfx = GCS_DATASET_PREFIX.strip().strip("/")
    if out_dir.is_dir() and pfx:
        archive = out_dir.parent / f"{out_dir.name}.tar.gz"
        if archive.exists():
            archive.unlink()
        try:
            run(["tar", "-czf", str(archive), "-C", str(out_dir.parent), out_dir.name])
            dst = f"gs://{GCS_BUCKET}/{pfx}/{archive.name}"
            run(["gsutil", "-m", "cp", str(archive), dst])
            print("Uploaded tile dataset archive:", dst)
            mpath = Path(LOCAL_RUN_DIR) / "run_config.json"
            if mpath.is_file():
                meta = json.loads(mpath.read_text(encoding="utf-8"))
                meta["dataset_archive_gcs"] = dst
                meta["dataset_archive_local_targz"] = str(archive)
                mpath.write_text(json.dumps(meta, indent=2), encoding="utf-8")
            try:
                os.remove(archive)
                print("Removed local", archive, "(dataset folder still on disk).")
            except OSError:
                pass
        except subprocess.CalledProcessError:
            print("Tile dataset GCS upload failed; data remains on local disk only. See error above.")

## After §7: keep your outputs

This notebook does not upload the run folder to GCS until you run the **next code cell**. It mirrors `LOCAL_RUN_DIR` to `GCS_RUN_URI` from §6.

You can also copy manually before the runtime disconnects:

- `gsutil -m cp -r LOCAL_RUN_DIR gs://BUCKET/your/prefix/run_name/`
- Cloud Storage browser in the Google Cloud Console
- Colab’s file pane to download smaller artifacts

The **tile dataset** is packed as one **`.tar.gz`** and uploaded with `gsutil cp` when `SYNC_DATASET_TO_GCS` is True and you used `create_doe_dataset.py`. To reuse it in a new session: `gsutil cp` the archive down, then `tar -xzf` into `DOE_DATASET_PATH` (or the path you pass to `doe_geoai.py`).


In [ ]:
from pathlib import Path

if "LOCAL_RUN_DIR" not in globals() or "GCS_RUN_URI" not in globals():
    raise RuntimeError("Run the persistent run directory cell first.")

if not Path(LOCAL_RUN_DIR).exists():
    raise FileNotFoundError(f"Missing local run directory: {LOCAL_RUN_DIR}")

run(["gsutil", "-m", "rsync", "-r", str(LOCAL_RUN_DIR), GCS_RUN_URI])
print("Synced:", GCS_RUN_URI)
